In [5]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from interpret.glassbox import ExplainableBoostingRegressor
from sklearn.datasets import fetch_openml, make_friedman1, make_friedman2, make_friedman3
from sklearn.exceptions import ConvergenceWarning
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from ucimlrepo import fetch_ucirepo
from xgboost import XGBRegressor

from spectral_paths.model import SpectralPathRegressor

warnings.filterwarnings("ignore", category=ConvergenceWarning, module="sklearn.neural_network")

_cwd = Path.cwd().resolve()
_examples_dir = (_cwd / "examples").resolve()


In [6]:
class RFF:
    """Random Fourier Features baseline with simple validation tuning."""

    def __init__(
        self,
        *,
        alpha_grid: tuple[float, ...] = (1e-2, 1e-1, 1.0),
        n_components_grid: tuple[int, ...] = (500, 1000),
        val_size: float = 0.2,
        random_state: int = 42,
    ) -> None:
        self.alpha_grid = alpha_grid
        self.n_components_grid = n_components_grid
        self.val_size = val_size
        self.random_state = random_state
        self.model = None
        self.best_params_: dict[str, float | int] | None = None

    def _gamma_grid(self, n_features: int) -> tuple[float, ...]:
        base_gamma = 1.0 / max(1, n_features)
        return (0.5 * base_gamma, base_gamma, 2.0 * base_gamma)

    def fit(self, X: np.ndarray, y: np.ndarray):
        X_tr, X_val, y_tr, y_val = train_test_split(
            X,
            y,
            test_size=self.val_size,
            random_state=self.random_state,
        )

        best_score = -np.inf
        best_model = None
        best_params = None
        gamma_grid = self._gamma_grid(X.shape[1])

        for gamma in gamma_grid:
            for alpha in self.alpha_grid:
                for n_components in self.n_components_grid:
                    candidate = make_pipeline(
                        StandardScaler(),
                        RBFSampler(
                            gamma=gamma,
                            n_components=n_components,
                            random_state=self.random_state,
                        ),
                        Ridge(alpha=alpha),
                    )
                    candidate.fit(X_tr, y_tr)
                    y_val_hat = candidate.predict(X_val)
                    score = float(r2_score(y_val, y_val_hat))

                    if score > best_score:
                        best_score = score
                        best_model = candidate
                        best_params = {
                            "gamma": gamma,
                            "alpha": alpha,
                            "n_components": n_components,
                        }

        if best_model is None or best_params is None:
            raise RuntimeError("RFF tuning failed to produce a fitted model.")

        final_model = make_pipeline(
            StandardScaler(),
            RBFSampler(
                gamma=float(best_params["gamma"]),
                n_components=int(best_params["n_components"]),
                random_state=self.random_state,
            ),
            Ridge(alpha=float(best_params["alpha"])),
        )
        final_model.fit(X, y)
        self.model = final_model
        self.best_params_ = best_params
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        if self.model is None:
            raise RuntimeError("Model must be fitted before calling predict.")
        return self.model.predict(X)


class SmallMLP:
    """Small MLP baseline with scaled features and simple validation tuning."""

    def __init__(
        self,
        *,
        hidden_layer_sizes_grid: tuple[tuple[int, ...], ...] = ((32,), (64,), (64, 32)),
        alpha_grid: tuple[float, ...] = (1e-4, 1e-3, 1e-2),
        val_size: float = 0.2,
        random_state: int = 42,
    ) -> None:
        self.hidden_layer_sizes_grid = hidden_layer_sizes_grid
        self.alpha_grid = alpha_grid
        self.val_size = val_size
        self.random_state = random_state
        self.model = None
        self.best_params_: dict[str, float | tuple[int, ...]] | None = None

    def fit(self, X: np.ndarray, y: np.ndarray):
        X_tr, X_val, y_tr, y_val = train_test_split(
            X,
            y,
            test_size=self.val_size,
            random_state=self.random_state,
        )

        best_score = -np.inf
        best_params = None

        for hidden_layer_sizes in self.hidden_layer_sizes_grid:
            for alpha in self.alpha_grid:
                candidate = make_pipeline(
                    StandardScaler(),
                    MLPRegressor(
                        hidden_layer_sizes=hidden_layer_sizes,
                        activation="relu",
                        solver="adam",
                        alpha=alpha,
                        learning_rate_init=1e-3,
                        max_iter=500,
                        early_stopping=True,
                        n_iter_no_change=20,
                        random_state=self.random_state,
                    ),
                )
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", ConvergenceWarning)
                    candidate.fit(X_tr, y_tr)
                y_val_hat = candidate.predict(X_val)
                score = float(r2_score(y_val, y_val_hat))

                if score > best_score:
                    best_score = score
                    best_params = {
                        "hidden_layer_sizes": hidden_layer_sizes,
                        "alpha": alpha,
                    }

        if best_params is None:
            raise RuntimeError("MLP tuning failed to produce a fitted model.")

        final_model = make_pipeline(
            StandardScaler(),
            MLPRegressor(
                hidden_layer_sizes=tuple(best_params["hidden_layer_sizes"]),
                activation="relu",
                solver="adam",
                alpha=float(best_params["alpha"]),
                learning_rate_init=1e-3,
                max_iter=500,
                early_stopping=True,
                n_iter_no_change=20,
                random_state=self.random_state,
            ),
        )
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            final_model.fit(X, y)
        self.model = final_model
        self.best_params_ = best_params
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        if self.model is None:
            raise RuntimeError("Model must be fitted before calling predict.")
        return self.model.predict(X)


def build_model(model_name: str, n_features: int):
    if model_name == "spectral_fast":
        return SpectralPathRegressor(
            max_paths=256,
            block_size=n_features,
            lambda_grid=list(np.logspace(-5, -1, 15)),
            scaler_type="robust_tanh",
            bound_percentiles=(5, 95),
            verbose=False,
            k_values=(1, 2, 3),
            early_stopping_patience=3,
            early_stopping_tol=1e-2,
        )
    if model_name == "spectral_slow":
        return SpectralPathRegressor(
            max_paths=1024,
            block_size=n_features,
            lambda_grid=list(np.logspace(-6, -1, 30)),
            scaler_type="robust_tanh",
            bound_percentiles=(5, 95),
            verbose=False,
            k_values=(1, 2, 3, 4),
            early_stopping_patience=8,
            early_stopping_tol=1e-4,
            greedy_subsample=5000,
        )
    if model_name == "xgboost":
        return XGBRegressor()
    if model_name == "rff":
        return RFF()
    if model_name == "mlp":
        return SmallMLP()
    if model_name == "ebm":
        return ExplainableBoostingRegressor()
    raise ValueError(f"Unsupported model: {model_name}")


def load_openml_dataset(dataset: dict[str, str | int]) -> tuple[np.ndarray, np.ndarray]:
    print(
        f"Fetching dataset '{dataset['name']}' from OpenML "
        f"(name={dataset['openml_name']}, version={dataset['version']})...",
        flush=True,
    )
    X, y = fetch_openml(
        name=dataset["openml_name"],
        version=dataset["version"],
        as_frame=False,
        parser="auto",
        return_X_y=True,
    )

    if dataset["openml_name"] == "slump":
        y = np.asarray(y, dtype=float)[:, 0].ravel()
    else:
        y = np.asarray(y, dtype=float).ravel()

    X_arr = np.asarray(X, dtype=float)
    print(
        f"Finished fetching '{dataset['name']}'. Rows: {X_arr.shape[0]}, features: {X_arr.shape[1]}",
        flush=True,
    )
    return X_arr, y


def load_pmlb_dataset(dataset: dict[str, str | int]) -> tuple[np.ndarray, np.ndarray]:
    def import_pmlb_fetch_data():
        for candidate in (_cwd, _examples_dir):
            candidate_str = str(candidate)
            if candidate_str in sys.path:
                sys.path.remove(candidate_str)

        shadowed_module = sys.modules.get("pmlb")
        shadowed_path = getattr(shadowed_module, "__file__", "") if shadowed_module else ""
        if shadowed_module and shadowed_path and Path(shadowed_path).resolve() == (_examples_dir / "pmlb.py"):
            del sys.modules["pmlb"]

        from pmlb import fetch_data as pmlb_fetch_data

        return pmlb_fetch_data

    fetch_data = import_pmlb_fetch_data()

    print(
        f"Fetching dataset '{dataset['name']}' from PMLB "
        f"(name={dataset['pmlb_name']})...",
        flush=True,
    )
    df = fetch_data(dataset["pmlb_name"], return_X_y=False)

    X = np.asarray(df.iloc[:, :-1].values, dtype=float)
    y = np.asarray(df.iloc[:, -1].values, dtype=float).ravel()

    print(
        f"Finished fetching '{dataset['name']}'. Rows: {X.shape[0]}, features: {X.shape[1]}",
        flush=True,
    )
    return X, y


def load_uci_dataset(dataset: dict[str, str | int]) -> tuple[np.ndarray, np.ndarray]:
    print(
        f"Fetching dataset '{dataset['name']}' from UCI ML Repo "
        f"(id={dataset['uci_id']})...",
        flush=True,
    )
    uci_dataset = fetch_ucirepo(id=int(dataset["uci_id"]))
    data = uci_dataset.data

    X = np.asarray(data.features, dtype=float)
    y_raw = data.targets
    if hasattr(y_raw, "to_numpy"):
        y_array = y_raw.to_numpy(dtype=float)
    else:
        y_array = np.asarray(y_raw, dtype=float)

    if y_array.ndim == 2:
        y = y_array[:, 0]
    else:
        y = y_array.ravel()

    y = np.asarray(y, dtype=float).ravel()
    print(
        f"Finished fetching '{dataset['name']}'. Rows: {X.shape[0]}, features: {X.shape[1]}",
        flush=True,
    )
    return X, y


def load_synthetic_dataset(dataset: dict[str, str | int | float]) -> tuple[np.ndarray, np.ndarray]:
    print(
        f"Generating synthetic dataset '{dataset['name']}' "
        f"(kind={dataset['generator']}, n_samples={dataset['n_samples']}, noise={dataset['noise']})...",
        flush=True,
    )
    random_state = int(dataset.get("random_state", 42))
    n_samples = int(dataset["n_samples"])
    noise = float(dataset.get("noise", 1.0))
    generator = str(dataset["generator"])

    if generator == "friedman1":
        X, y = make_friedman1(n_samples=n_samples, n_features=10, noise=noise, random_state=random_state)
    elif generator == "friedman2":
        X, y = make_friedman2(n_samples=n_samples, noise=noise, random_state=random_state)
    elif generator == "friedman3":
        X, y = make_friedman3(n_samples=n_samples, noise=noise, random_state=random_state)
    else:
        raise ValueError(f"Unsupported synthetic generator: {generator}")

    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).ravel()
    print(
        f"Finished generating '{dataset['name']}'. Rows: {X.shape[0]}, features: {X.shape[1]}",
        flush=True,
    )
    return X, y


def load_dataset(dataset: dict[str, str | int]) -> tuple[np.ndarray, np.ndarray]:
    if dataset["source"] == "openml":
        return load_openml_dataset(dataset)
    if dataset["source"] == "pmlb":
        return load_pmlb_dataset(dataset)
    if dataset["source"] == "uci":
        return load_uci_dataset(dataset)
    if dataset["source"] == "synthetic":
        return load_synthetic_dataset(dataset)
    raise ValueError(f"Unsupported dataset source: {dataset['source']}")


def timed_fit_predict(model_name: str, X_tr: np.ndarray, y_tr: np.ndarray, X_te: np.ndarray):
    model = build_model(model_name, X_tr.shape[1])

    fit_start = time.perf_counter()
    model.fit(X_tr, y_tr)
    fit_time = time.perf_counter() - fit_start

    predict_start = time.perf_counter()
    y_hat = model.predict(X_te)
    predict_time = time.perf_counter() - predict_start

    return y_hat, fit_time, predict_time


In [7]:
datasets: list[dict[str, str | int | float]] = [
    {"source": "openml", "name": "Concrete Slump", "openml_name": "slump", "version": 2},
    {"source": "openml", "name": "Yacht Hydrodynamics", "openml_name": "yacht_hydrodynamics", "version": 1},
    {"source": "openml", "name": "Cancer Drug Response", "openml_name": "Cancer_Drug_Response", "version": 1},
    {"source": "openml", "name": "Aquatic Toxicity", "openml_name": "qsar_aquatic_toxicity", "version": 1},
    {"source": "openml", "name": "Izmir Weather", "openml_name": "weather_izmir", "version": 1},
    {"source": "openml", "name": "Ankara Weather", "openml_name": "weather_ankara", "version": 1},
    {"source": "openml", "name": "airfoil", "openml_name": "airfoil_self_noise", "version": 8},
    {"source": "pmlb", "name": "Echo Cardiogram", "pmlb_name": "1199_BNG_echoMonths"},
    {"source": "pmlb", "name": "Wind Speed", "pmlb_name": "503_wind"},
    {"source": "pmlb", "name": "CPU Utilisation", "pmlb_name": "197_cpu_act"},
    {"source": "uci", "name": "Energy Efficiency", "uci_id": 242},
    {"source": "uci", "name": "Concrete Compressive Strength", "uci_id": 165},
    {"source": "uci", "name": "Wine Quality", "uci_id": 186},
    {"source": "uci", "name": "Superconductivity", "uci_id": 464},
    {"source": "uci", "name": "Bike Sharing", "uci_id": 275},
    {"source": "uci", "name": "Student Performance", "uci_id": 320},
    {"source": "uci", "name": "Obesity Levels", "uci_id": 544},
    {"source": "uci", "name": "Energy Appliances", "uci_id": 374},
    {"source": "synthetic", "name": "Friedman #1", "generator": "friedman1", "n_samples": 1000, "noise": 1.0, "random_state": 42},
    {"source": "synthetic", "name": "Friedman #2", "generator": "friedman2", "n_samples": 1000, "noise": 1.0, "random_state": 42},
    {"source": "synthetic", "name": "Friedman #3", "generator": "friedman3", "n_samples": 1000, "noise": 1.0, "random_state": 42},
]

SEEDS = [1, 2, 3, 4, 5]
MODELS = ["spectral_fast", "spectral_slow", "xgboost", "rff", "mlp"]
results_records: list[dict[str, float | int | str]] = []


In [8]:
for dataset in datasets:
    print(f"\nStarting dataset: {dataset['name']}", flush=True)
    X, y = load_dataset(dataset)
    print(f"\nDataset: {dataset['name']} ({dataset['source']})")
    print(f"Rows: {X.shape[0]}, features: {X.shape[1]}")
    print(f"{'Model':<14} {'Mean R2':>10} {'Std R2':>10} {'Fit (s)':>10} {'Pred (s)':>10}")

    for model_name in MODELS:
        r2_scores: list[float] = []
        fit_times: list[float] = []
        predict_times: list[float] = []

        for seed in SEEDS:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X,
                y,
                test_size=0.20,
                random_state=seed,
            )

            y_hat, fit_time, predict_time = timed_fit_predict(model_name, X_tr, y_tr, X_te)
            r2_value = float(r2_score(y_te, y_hat))

            r2_scores.append(r2_value)
            fit_times.append(fit_time)
            predict_times.append(predict_time)
            results_records.append(
                {
                    "source": str(dataset["source"]),
                    "dataset": str(dataset["name"]),
                    "model": model_name,
                    "seed": seed,
                    "n_rows": int(X.shape[0]),
                    "n_features": int(X.shape[1]),
                    "r2": r2_value,
                    "fit_time_sec": fit_time,
                    "predict_time_sec": predict_time,
                }
            )

        print(
            f"{model_name:<14} "
            f"{np.mean(r2_scores):>10.4f} "
            f"{np.std(r2_scores):>10.4f} "
            f"{np.mean(fit_times):>10.3f} "
            f"{np.mean(predict_times):>10.4f}"
        )

results_df = pd.DataFrame(results_records)
summary_df = (
    results_df.groupby(["source", "dataset", "model"], as_index=False)
    .agg(
        mean_r2=("r2", "mean"),
        std_r2=("r2", "std"),
        mean_fit_time_sec=("fit_time_sec", "mean"),
        mean_predict_time_sec=("predict_time_sec", "mean"),
    )
    .sort_values(["source", "dataset", "mean_r2"], ascending=[True, True, False])
    .reset_index(drop=True)
)



Starting dataset: Concrete Slump
Fetching dataset 'Concrete Slump' from OpenML (name=slump, version=2)...
Finished fetching 'Concrete Slump'. Rows: 103, features: 7

Dataset: Concrete Slump (openml)
Rows: 103, features: 7
Model             Mean R2     Std R2    Fit (s)   Pred (s)
spectral_fast      0.1184     0.2054      0.165     0.0002
spectral_slow      0.0503     0.2640      0.205     0.0001
xgboost            0.1272     0.3320      0.098     0.0012
rff                0.2736     0.1326      0.121     0.0011
mlp               -0.1357     0.2770      2.497     0.0004

Starting dataset: Yacht Hydrodynamics
Fetching dataset 'Yacht Hydrodynamics' from OpenML (name=yacht_hydrodynamics, version=1)...
Finished fetching 'Yacht Hydrodynamics'. Rows: 308, features: 6

Dataset: Yacht Hydrodynamics (openml)
Rows: 308, features: 6
Model             Mean R2     Std R2    Fit (s)   Pred (s)
spectral_fast      0.9858     0.0076      0.103     0.0002
spectral_slow      0.9977     0.0016      1.395 

KeyboardInterrupt: 

In [ ]:
results_df


,source,dataset,model,seed,n_rows,n_features,r2,fit_time_sec,predict_time_sec
0,openml,Concrete Slump,spectral_fast,1,103,7,0.335948,0.130190,0.000149
1,openml,Concrete Slump,spectral_fast,2,103,7,0.272388,0.086831,0.000103
2,openml,Concrete Slump,spectral_fast,3,103,7,-0.252957,0.085058,0.000124
3,openml,Concrete Slump,spectral_fast,4,103,7,0.086740,0.093420,0.000211
4,openml,Concrete Slump,spectral_fast,5,103,7,0.149795,0.211207,0.000151
...,...,...,...,...,...,...,...,...,...
175,pmlb,CPU Utilisation,rff,1,8192,21,-0.052801,0.422731,0.119207
176,pmlb,CPU Utilisation,rff,2,8192,21,-0.082394,0.232340,0.094552
177,pmlb,CPU Utilisation,rff,3,8192,21,-0.053323,0.385800,0.125786
178,pmlb,CPU Utilisation,rff,4,8192,21,-0.124040,0.444188,0.068115


In [ ]:
results_df.to_csv("results.csv")